In [2]:
%pip install pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

In [4]:
# Recreating the clean dataset as created in '03-data-merge-and-cleaning.ipynb'

amazon1 = pd.read_csv('/workspaces/group-project-bas-team/data/AmazonData1.csv')
amazon2 = pd.read_csv('/workspaces/group-project-bas-team/data/AmazonData2.csv')
amazon_data = [amazon1, amazon2]
amazon = pd.concat(amazon_data)

survey = pd.read_csv('/workspaces/group-project-bas-team/data/survey.csv')
survey = survey.sample(800, random_state=42) 
# 800 surveys keeps our final dataset under 100mb, allowing us to save it to our data folder instead of repeating all our cleaning and transformation steps

data = pd.merge(amazon, survey, on='Survey ResponseID', how='inner')

codes = data[['ASIN/ISBN (Product Code)', 'Title']].drop_duplicates().dropna()
codes_dict = codes.set_index('ASIN/ISBN (Product Code)')['Title'].to_dict()
data['Title'] = data['ASIN/ISBN (Product Code)'].map(codes_dict)

cats = data[['ASIN/ISBN (Product Code)', 'Category']].drop_duplicates().dropna()
cats_dict = cats.set_index('ASIN/ISBN (Product Code)')['Category'].to_dict()
data['Category'] = data['ASIN/ISBN (Product Code)'].map(cats_dict)

data['Q-life-changes'] = data['Q-life-changes'].fillna('None')

data = data.dropna()

data.columns = data.columns.str.replace('Q-', '')\
    .str.replace('demos-', '')\
    .str.replace('amazon-use-', '')\
    .str.replace('substance-use-', '')\
    .str.replace('personal-', '')

data = data.reset_index()
data = data.drop('index', axis=1)
data.info()


<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Order Date                157026 non-null  str    
 1   Purchase Price Per Unit   157026 non-null  float64
 2   Quantity                  157026 non-null  int64  
 3   Shipping Address State    157026 non-null  str    
 4   Title                     157026 non-null  str    
 5   ASIN/ISBN (Product Code)  157026 non-null  str    
 6   Category                  157026 non-null  str    
 7   Survey ResponseID         157026 non-null  str    
 8   age                       157026 non-null  str    
 9   hispanic                  157026 non-null  str    
 10  race                      157026 non-null  str    
 11  education                 157026 non-null  str    
 12  income                    157026 non-null  str    
 13  gender                    157026 non-null  str    
 14 

In [5]:
# Exploding columns with multiple values ('life-changes' and 'race')
# NO LONGER EXPLODING COLUMNS - INTERSECTIONALITY MAY BE IMPORTANT

# data['race'] = data['race'].str.split(',')
# data['life-changes'] = data['life-changes'].str.split(',')

In [6]:
# data = data.explode('race')
# data = data.explode('life-changes')

In [7]:
# separating dates into month, day, and year columns
data[['order_month', 'order_day', 'order_year']] = data['Order Date'].str.split('/', expand=True)

data = data.drop('Order Date', axis=1)

data[['order_month', 'order_day', 'order_year']] = data[['order_month', 'order_day', 'order_year']].apply(pd.to_numeric)

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 32 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Purchase Price Per Unit   157026 non-null  float64
 1   Quantity                  157026 non-null  int64  
 2   Shipping Address State    157026 non-null  str    
 3   Title                     157026 non-null  str    
 4   ASIN/ISBN (Product Code)  157026 non-null  str    
 5   Category                  157026 non-null  str    
 6   Survey ResponseID         157026 non-null  str    
 7   age                       157026 non-null  str    
 8   hispanic                  157026 non-null  str    
 9   race                      157026 non-null  str    
 10  education                 157026 non-null  str    
 11  income                    157026 non-null  str    
 12  gender                    157026 non-null  str    
 13  sexual-orientation        157026 non-null  str    
 14 

In [8]:
# Encoding Categorical Data
# Dropping features not relevant to predicting purchase behavior

cols_to_drop = [
    'Survey ResponseID',
    'sell-YOUR-data',
    'sell-consumer-data',
    'small-biz-use',
    'census-use',
    'research-society'
]

data = data.drop(columns=cols_to_drop)



In [9]:
# Ordinal encoding for columns where the categories have an order
ordinal_mappings = {
    'age': {
        '18 - 24 years': 1,
        '25 - 34 years': 2,
        '35 - 44 years': 3,
        '45 - 54 years': 4,
        '55 - 64 years': 5,
        '65 and older': 6
    },
    'education': {
        'Prefer not to say': 0,
        'Some high school or less': 1,
        'High school diploma or GED': 2,
        "Bachelor's degree": 3,
        'Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS, etc)': 4
    },
    'income': {
        'Prefer not to say': 0,
        'Less than $25,000': 1,
        '$25,000 - $49,999': 2,
        '$50,000 - $74,999': 3,
        '$75,000 - $99,999': 4,
        '$100,000 - $149,999': 5,
        '$150,000 or more': 6
    },
    'howmany': {
        '1 (just me!)': 1,
        '2': 2,
        '3': 3,
        '4+': 4
    },
    'hh-size': {
        '1 (just me!)': 1,
        '2': 2,
        '3': 3,
        '4+': 4
    },
    'how-oft': {
        'Less than 5 times per month': 1,
        '5 - 10 times per month': 2,
        'More than 10 times per month': 3
    }
}

for col, mapping in ordinal_mappings.items():
    data[col] = data[col].map(mapping)
    unmapped = data[col].isna().sum()
    if unmapped > 0:
        print(f"Warning: {unmapped} unmapped values in '{col}' — check category strings match exactly")

print("Ordinal encoding complete.")
data[list(ordinal_mappings.keys())].head()

Ordinal encoding complete.


,age,education,income,howmany,hh-size,how-oft
0,3,3,2,1,1,1
1,3,3,2,1,1,1
2,3,3,2,1,1,1
3,3,3,2,1,1,1
4,3,3,2,1,1,1


In [10]:
# Binary encoding
# hispanic is our only true binary column

yes_no_map = {'Yes': 1, 'No': 0}

data['hispanic'] = data['hispanic'].map(yes_no_map)

In [11]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 26 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Purchase Price Per Unit   157026 non-null  float64
 1   Quantity                  157026 non-null  int64  
 2   Shipping Address State    157026 non-null  str    
 3   Title                     157026 non-null  str    
 4   ASIN/ISBN (Product Code)  157026 non-null  str    
 5   Category                  157026 non-null  str    
 6   age                       157026 non-null  int64  
 7   hispanic                  157026 non-null  int64  
 8   race                      157026 non-null  str    
 9   education                 157026 non-null  int64  
 10  income                    157026 non-null  int64  
 11  gender                    157026 non-null  str    
 12  sexual-orientation        157026 non-null  str    
 13  state                     157026 non-null  str    
 14 

In [12]:
# One hot encoding
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

data_one_hot = encoder.fit_transform(data[['cigarettes', 'alcohol', 'marijuana', 'diabetes',\
                                            'wheelchair', 'Shipping Address State', 'gender',\
                                            'sexual-orientation', 'state', 'race', 'life-changes']])
df_one_hot = pd.DataFrame(data_one_hot, columns=encoder\
                          .get_feature_names_out(['cigarettes', 'alcohol', 'marijuana', 'diabetes',\
                                                    'wheelchair', 'Shipping Address State', 'gender',\
                                                    'sexual-orientation', 'state', 'race', 'life-changes']))

In [13]:
# Remove one column from each variable to prevent multicollinearity
df_one_hot = df_one_hot.drop(columns=['cigarettes_Prefer not to say', 'alcohol_Prefer not to say', \
                                      'marijuana_Prefer not to say', 'diabetes_Prefer not to say', \
                                        'wheelchair_Prefer not to say', 'Shipping Address State_HI', \
                                        'gender_Prefer not to say', 'sexual-orientation_prefer not to say', \
                                        'state_Alaska', 'race_Other', 'life-changes_None'])

In [14]:
# Add df_one_hot to existing data and remove original columns
df_combined = pd.concat([data, df_one_hot], axis=1)
df_combined = df_combined.drop(columns=['cigarettes', 'alcohol', 'marijuana', 'diabetes',\
                                            'wheelchair', 'Shipping Address State', 'gender',\
                                            'sexual-orientation', 'state', 'race', 'life-changes'])
df_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 156 entries, Purchase Price Per Unit to life-changes_Moved place of residence,Had a child
dtypes: float64(142), int64(11), str(3)
memory usage: 186.9 MB


In [15]:
df_combined.head()

,Purchase Price Per Unit,Quantity,Title,ASIN/ISBN (Product Code),Category,age,hispanic,education,income,howmany,...,"life-changes_Lost a job ,Divorce","life-changes_Lost a job ,Divorce,Moved place of residence","life-changes_Lost a job ,Had a child","life-changes_Lost a job ,Moved place of residence","life-changes_Lost a job ,Moved place of residence,Became pregnant","life-changes_Lost a job ,Moved place of residence,Became pregnant,Had a child","life-changes_Lost a job ,Moved place of residence,Had a child",life-changes_Moved place of residence,"life-changes_Moved place of residence,Became pregnant,Had a child","life-changes_Moved place of residence,Had a child"
0,7.98,1,SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...,B0143RTB1E,FLASH_MEMORY,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13.99,1,Betron BS10 Earphones Wired Headphones in Ear ...,B01MA1MJ6H,HEADPHONES,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10.45,1,Perfecto Stainless Steel Shaving Bowl. Durable...,B06XWF9HML,DISHWARE_BOWL,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,10.00,1,Proraso Shaving Cream for Men,B00837ZOI0,SHAVING_AGENT,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10.99,1,Micro USB Cable Android Charger - Syncwire [2-...,B01GFB2E9M,COMPUTER_PROCESSOR,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# label encode Title, ASIN/ISBN (Product Code), and category, since these will be our potential target values
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

target_cols = ['Title', 'ASIN/ISBN (Product Code)', 'Category']

for col in target_cols:
    df_combined[col] = le.fit_transform(df_combined[col].astype(str))

In [19]:
df_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 156 entries, Purchase Price Per Unit to life-changes_Moved place of residence,Had a child
dtypes: float64(142), int64(14)
memory usage: 186.9 MB


In [69]:
df_combined.to_csv('/workspaces/group-project-bas-team/data/cleaned_data.csv', index=False)

In [70]:
# Binary encoding for Yes/No columns

# binary_cols = [
#     'hispanic',
#     'cigarettes',
#     'alcohol',
#     'diabetes',
#     'wheelchair'
# ]

# Preview unique values first to confirm mapping
# for col in binary_cols:
#     print(f"{col}: {data[col].unique()}")

In [71]:
# Mapping Yes/No to 1/0

# yes_no_map = {'Yes': 1, 'No': 0}

# for col in ['hispanic', 'diabetes', 'wheelchair']:
#     data[col] = data[col].map(yes_no_map)

# Substance use columns have more nuanced values — encode as current user (1) or not (0)
# substance_map = {
#     'Yes': 1,
#     'No': 0,
#     'I stopped in the recent past': 0,
#     'I never did this': 0
# }

# for col in ['cigarettes', 'alcohol', 'marijuana']:
#     data[col] = data[col].map(substance_map)


In [72]:
# Label encoding for nominal (unordered) categorical columns
# These have no inherent order, so we use integer codes assigned alphabetically

# from sklearn.preprocessing import LabelEncoder

# nominal_cols = [
#     'Shipping Address State',
#     'Title',
#     'ASIN/ISBN (Product Code)',
#     'Category',
#     'gender',
#     'sexual-orientation',
#     'state',
#     'race',
#     'life-changes'
# ]

# le = LabelEncoder()
# label_encoders = {}  # store encoders in case we need to inverse_transform later

# for col in nominal_cols:
#     data[col] = le.fit_transform(data[col].astype(str))
#     label_encoders[col] = le
#     print(f"Encoded '{col}' — {data[col].nunique()} unique values")

In [73]:
# Final check — confirm all columns are numeric

# data.info()
# data.describe()
